# 04 — Virtual screening: dock library + generated SMILES against MCAS targets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrdulasolutions/MCAS.Opensource/blob/main/notebooks/04_virtual_screening.ipynb)

> ⚠️ **Not medical advice.** Research/hypothesis use only.

What this notebook does:
1. Reads target UniProt IDs from `data/targets/MCAS_Targets.csv`.
2. Downloads AlphaFold predicted structures for each target.
3. Docks the master library + REINVENT outputs with **DiffDock** (Colab) or a CPU fallback (`smina`).
4. Writes per-target docking scores to `outputs/docking_<target>.csv`.

**Recommended runtime:** Colab GPU for DiffDock. For CPU, use the smina fallback cell.

In [ ]:
from pathlib import Path
import urllib.request, os
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
TARGETS_CSV = REPO_ROOT / 'data' / 'targets' / 'MCAS_Targets.csv'
LIBRARY_SDF = REPO_ROOT / 'outputs' / 'library.sdf'
OUT_DIR = REPO_ROOT / 'outputs'
STRUCT_DIR = OUT_DIR / 'structures'
STRUCT_DIR.mkdir(parents=True, exist_ok=True)

targets = pd.read_csv(TARGETS_CSV)
targets[['gene_symbol', 'uniprot_id', 'role_in_MCAS']]

In [ ]:
def download_alphafold(uniprot_id: str) -> Path:
    url = f'https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb'
    dest = STRUCT_DIR / f'{uniprot_id}.pdb'
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

# Demo: pull MRGPRX2 and KIT structures (primary remission + clonal targets)
PRIMARY_TARGETS = ['MRGPRX2', 'KIT', 'NFE2L2']
for t in PRIMARY_TARGETS:
    uid = targets.loc[targets['gene_symbol'] == t, 'uniprot_id'].iloc[0]
    p = download_alphafold(uid)
    print(t, '->', p)

## DiffDock path (Colab GPU recommended)

DiffDock does blind docking — no pocket definition required. Install in Colab:

```bash
!git clone https://github.com/gcorso/DiffDock.git
%cd DiffDock
!pip install -r requirements.txt
```

Then run inference per (protein, ligand) pair from `outputs/library.sdf`. The DiffDock README has the up-to-date CLI; we wire it here.

In [ ]:
# Pseudo-driver; fill in once DiffDock is installed in your Colab session.
import subprocess

def diffdock_dock(protein_pdb, ligand_sdf, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        'python', '-m', 'inference',
        '--protein_path', str(protein_pdb),
        '--ligand', str(ligand_sdf),
        '--out_dir', str(out_dir),
        '--inference_steps', '20',
        '--samples_per_complex', '10',
    ]
    print('would run:', ' '.join(cmd))
    # subprocess.run(cmd, check=True, cwd='DiffDock')

# Example: dock library against MRGPRX2
uid = targets.loc[targets['gene_symbol'] == 'MRGPRX2', 'uniprot_id'].iloc[0]
diffdock_dock(STRUCT_DIR / f'{uid}.pdb', LIBRARY_SDF, OUT_DIR / 'dock_MRGPRX2')

## CPU fallback: smina (AutoDock Vina fork)

`smina` works on Mac CPU. Install via conda or homebrew. Define a binding box around the predicted pocket (use `prody` or `fpocket` to identify candidate pockets if you don't have one).

In [ ]:
# Conceptual smina batch driver — fill in pocket coords per target before running.
POCKET_BOX = {  # placeholder; replace with actual pocket centers (Å)
    'MRGPRX2': dict(center=(0, 0, 0), size=(22, 22, 22)),
    'KIT':     dict(center=(0, 0, 0), size=(22, 22, 22)),
    'NFE2L2':  dict(center=(0, 0, 0), size=(22, 22, 22)),
}

def smina_dock(protein_pdb, ligand_sdf, target_name, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    box = POCKET_BOX[target_name]
    cmd = [
        'smina',
        '--receptor', str(protein_pdb),
        '--ligand', str(ligand_sdf),
        '--center_x', str(box['center'][0]),
        '--center_y', str(box['center'][1]),
        '--center_z', str(box['center'][2]),
        '--size_x', str(box['size'][0]),
        '--size_y', str(box['size'][1]),
        '--size_z', str(box['size'][2]),
        '--out', str(out_dir / f'{target_name}_docked.sdf'),
        '--num_modes', '5',
    ]
    print('would run:', ' '.join(cmd))
    # subprocess.run(cmd, check=True)

## After docking — write per-target score CSVs

Parse the docking outputs into `outputs/docking_<target>.csv` with columns `name, target, score, pose_path`. `05_hypothesis_ranking.ipynb` will pick these up.